# Milestone 16 Objective

Integrate selected MCP tools into existing agents in a safe, optional way.

## LangGraph State vs MCP

LangGraph State remains the shared workflow memory. MCP is only the standardized tool-access layer.

## MCP Integration Strategy

Local tools remain the default. MCP is enabled only with `use_mcp_tools=True`, and agents fall back to local tools if MCP is unavailable.

## Integrated Agents

- Repository Analyzer
- API Testing Agent
- Report Agent

## Main Workflow

The main workflow remains unchanged:

`START -> orchestrator -> repo_analyzer -> rag -> test_planner -> api_testing -> bug_analysis -> report -> END`

## Security Note

MCP config has no secrets. Tokens and sensitive values are masked before display or persistence.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root for MCP server imports:", project_root)


Project root for MCP server imports: C:\Users\malak\Desktop\TEST_AUTO\sma_test_automation


In [2]:
# Part A - MCP self-test
from mcp_servers.testing_tools_server import health_check

health_check()

{'status': 'ok', 'server': 'testing-tools-server', 'tools_version': '0.1.0'}

In [3]:
# Part B - should_use_mcp behavior
from test_auto.mcp.tool_router import should_use_mcp

{
    "default": should_use_mcp(),
    "from_user_preferences": should_use_mcp({"use_mcp_tools": True}),
    "from_config": should_use_mcp(config={"mcp": {"enabled": True}}),
}

{'default': False, 'from_user_preferences': True, 'from_config': True}

In [4]:
# Part C - fake/local Repository Analyzer call with MCP disabled/enabled
from pathlib import Path
from tempfile import TemporaryDirectory
from test_auto.agents.repo_analyzer import analyze_repository

with TemporaryDirectory() as tmp:
    repo = Path(tmp) / "fake_repo"
    (repo / "todo").mkdir(parents=True)
    (repo / "templates").mkdir(parents=True)
    (repo / "README.md").write_text("# Fake Django REST API", encoding="utf-8")
    (repo / "requirements.txt").write_text("django\ndjangorestframework\npytest\n", encoding="utf-8")
    (repo / "manage.py").write_text("# placeholder", encoding="utf-8")
    (repo / "todo" / "urls.py").write_text("from django.urls import path\nurlpatterns = []\n", encoding="utf-8")
    (repo / "templates" / "login.html").write_text("<form>login</form>", encoding="utf-8")

    local_result = analyze_repository(repo_path=str(repo))
    mcp_result = analyze_repository(repo_path=str(repo), user_preferences={"use_mcp_tools": True})

    {
        "local_backend": local_result["agent_output"]["metadata"].get("tool_backend"),
        "mcp_backend": mcp_result["agent_output"]["metadata"].get("tool_backend"),
        "mcp_fallback_used": mcp_result["agent_output"]["metadata"].get("mcp_fallback_used"),
    }

In [5]:
# Part D - API agent with MCP enabled and safe fallback if MCP is unavailable
from test_auto.agents.api_testing_agent import run_api_testing_agent_alone

api_result = run_api_testing_agent_alone(
    target_url="http://127.0.0.1:9",
    test_plan={
        "api_tests": [
            {
                "id": "API_001",
                "name": "List todos",
                "method": "GET",
                "endpoint": "/api/todos/",
                "expected_status": 200,
                "assertions": [],
            }
        ]
    },
    user_preferences={"use_mcp_tools": True, "api_timeout_seconds": 1},
    allow_mutating=False,
)

api_result["agent_output"]["metadata"]

{'target_url': 'http://127.0.0.1:9',
 'auth_token_used': False,
 'allow_mutating_api_tests': False,
 'timeout_seconds': 1,
 'tool_backend': 'mixed',
 'mcp_fallback_used': True,
 'mcp_events': [{'tool_name': 'send_http_request_tool',
   'used_mcp': False,
   'fallback_used': True,
   'error': 'Cannot use sync MCP invocation inside an active event loop; use async version.'}]}

In [6]:
# Part E - full workflow with fake repo and MCP enabled
from pathlib import Path
from tempfile import TemporaryDirectory
from test_auto.graph.workflow import run_workflow

with TemporaryDirectory() as tmp:
    repo = Path(tmp) / "fake_repo"
    (repo / "todo").mkdir(parents=True)
    (repo / "templates").mkdir(parents=True)
    (repo / "tests").mkdir(parents=True)
    (repo / "README.md").write_text("# Fake Django REST API", encoding="utf-8")
    (repo / "requirements.txt").write_text("django\ndjangorestframework\npytest\n", encoding="utf-8")
    (repo / "manage.py").write_text("# placeholder", encoding="utf-8")
    (repo / "todo" / "urls.py").write_text("from django.urls import path\nurlpatterns = [path('api/todos/', object())]\n", encoding="utf-8")
    (repo / "todo" / "views.py").write_text("from rest_framework.decorators import api_view\n@api_view(['GET'])\ndef todos(request): pass\n", encoding="utf-8")
    (repo / "templates" / "login.html").write_text("<form>login</form>", encoding="utf-8")
    (repo / "tests" / "test_todo_api.py").write_text("def test_ok(): assert True\n", encoding="utf-8")

    final_state = run_workflow(
        {
            "repo_path": str(repo),
            "target_url": "http://127.0.0.1:9",
            "user_preferences": {
                "test_types": ["api"],
                "execution_mode": "sequential",
                "focus": "JWT authentication todo CRUD API tests",
                "planner_use_llm": False,
                "use_mcp_tools": True,
                "api_timeout_seconds": 1,
            },
            "errors": [],
            "agent_logs": [],
        }
    )

    [
        {
            "agent": log.get("agent"),
            "metadata": log.get("metadata"),
        }
        for log in final_state.get("agent_logs", [])
        if isinstance(log, dict) and (log.get("metadata") or {}).get("tool_backend")
    ]